# ETL Bronze - Precipitacion Salto Grande

Carga los JSON diarios de SG en una tabla Bronze idempotente.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructField, StructType
from pyspark.sql.window import Window

BRONZE_TABLE = 'weather.bronze.sg_rainfall'
RAW_PATH = '/Volumes/weather/raw/sg_volume/json/daily/'

schema = StructType(
    [
        StructField('Fecha', StringType(), True),
        StructField('Id_Estacion', StringType(), True),
        StructField('P', DoubleType(), True),
        StructField('Nombre', StringType(), True),
        StructField('Latitud', DoubleType(), True),
        StructField('Longitud', DoubleType(), True),
        StructField('source_api', StringType(), True),
        StructField('extracted_at', StringType(), True),
    ]
)

In [ ]:
try:
    raw_files = [item.path for item in dbutils.fs.ls(RAW_PATH) if item.path.endswith('.json')]
except Exception:
    raw_files = []

if not raw_files:
    dbutils.notebook.exit(f'No SG raw JSON files found at {RAW_PATH}')

raw_df = spark.read.schema(schema).option('multiLine', True).json(RAW_PATH)

bronze_df = (
    raw_df.withColumn('fecha', F.to_date(F.substring(F.col('Fecha'), 1, 10)))
    .withColumn('fecha_raw', F.col('Fecha').cast('string'))
    .withColumn('id_estacion', F.col('Id_Estacion').cast('string'))
    .withColumn('nombre', F.col('Nombre').cast('string'))
    .withColumn('latitud', F.col('Latitud').cast('double'))
    .withColumn('longitud', F.col('Longitud').cast('double'))
    .withColumn('p', F.col('P').cast('double'))
    .withColumn('source_api', F.coalesce(F.col('source_api'), F.lit('salto_grande_hidroserie_historica')))
    .withColumn('source_file', F.col('_metadata.file_path'))
    .withColumn('extracted_at', F.to_timestamp('extracted_at'))
    .withColumn('ingestion_date', F.current_date())
    .withColumn('loaded_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .filter(F.col('fecha').isNotNull())
    .filter(F.col('id_estacion').isNotNull())
)

window = Window.partitionBy('fecha', 'id_estacion').orderBy(F.col('extracted_at').desc_nulls_last(), F.col('source_file').desc())
bronze_df = (
    bronze_df.withColumn('row_number', F.row_number().over(window))
    .filter(F.col('row_number') == 1)
    .drop('row_number')
    .select(
        'fecha', 'fecha_raw', 'id_estacion', 'nombre', 'latitud', 'longitud', 'p',
        'source_api', 'source_file', 'extracted_at', 'ingestion_date', 'loaded_at', 'updated_at'
    )
)

if bronze_df.limit(1).count() == 0:
    dbutils.notebook.exit('No valid SG rows to merge')

DeltaTable.forName(spark, BRONZE_TABLE).alias('t').merge(
    bronze_df.alias('s'),
    't.fecha = s.fecha AND t.id_estacion = s.id_estacion',
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

spark.table(BRONZE_TABLE).agg(F.min('fecha').alias('inicio'), F.max('fecha').alias('fin'), F.count('*').alias('rows')).show()